# Either listed role is sufficient

A manager or an auditor may call the operation. Neither role inherits the other.

Run with the Python environment of this checkout (`.venv`). These examples use the API in this branch; installing a released package may give a different API.

[Script](15_alternative_roles.py) · [Tutorial](../../docs/tutorials/step-03-authorization-and-roles.md)

## Imports

In [1]:
"""Either listed role is sufficient."""

from __future__ import annotations

import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## The rule

A manager or an auditor may call the operation. Neither role inherits the other.

In [2]:
class StoreDomain(BaseDomain):
    """Group the order operations."""

    name = "store"
    description = "Order management"


class ManagerRole(ApplicationRole):
    """Permit order management."""

    name = "manager"
    description = "Can manage orders"


class OrderParams(BaseParams):
    """Identify the order to view."""

    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    """Report the requested order."""

    order_id: str = Field(description="Requested order identifier")


class AuditorRole(ApplicationRole):
    """Permit reviewing order operations."""

    name = "auditor"
    description = "Can review orders"


@meta(description="View an order", domain=StoreDomain)
@check_roles([ManagerRole, AuditorRole])
class ViewOrderAction(BaseAction[OrderParams, OrderResult]):
    """View an order after checking access."""

    @summary_aspect("View the order")
    async def view_summary(self, params, state, box, connections):
        """Return the order identifier."""
        return OrderResult(order_id=params.order_id)

## Try it

Run the cells in order. The calls below are the same as in the script.

In [3]:
machine = ActionProductMachine(cache_coordinator=None)
manager = Context(user=UserInfo(user_id="m-1", roles=(ManagerRole,)))
auditor = Context(user=UserInfo(user_id="a-1", roles=(AuditorRole,)))
params = OrderParams(order_id="ord-001")

In [4]:
answer = await machine.check_access_decide(manager, ViewOrderAction, params)
print("manager:", answer.kind)
answer = await machine.check_access_decide(auditor, ViewOrderAction, params)
print("auditor:", answer.kind)
answer = await machine.check_access_decide(Context(), ViewOrderAction, params)
print("anonymous:", answer.model_dump(mode="json"))

manager: allowed
auditor: allowed
anonymous: {'kind': 'refused', 'gate': 'CHECK_ROLES', 'reason': None}


## What the result means

Each listed role is sufficient on its own. The anonymous caller has neither and is refused at CHECK_ROLES. A list declares alternatives, not a requirement to hold both roles.